## HMM Distillation Tutorial (For VLM)

To distill a hmm model, we need to 
* (1) prepare the input data for sampling
* (2) sample the output
* (3) Initialize the checkpoint-0 with lvd
* (4) train the hmm model.

### Basic Config

In [ ]:
import os

# Sampling Dataset Path: The path to save the dataset used for sampling
SAMPLING_DATASET_PATH = "openvlthinker_hard_boxed.json"
# SAMPLING_DATASET_PATH = "openvlthinker_medium_boxed.json"

# Model path or model name from Hugging Face
# BASE_MODEL_PATH = "ydeng9/OpenVLThinker-7B-v1.2"
# BASE_MODEL_PATH = "Qwen/Qwen2.5-VL-7B-Instruct"
BASE_MODEL_PATH = "JeffreyZLuo/Qwen2.5-7B-15-Medium"
# The name used to save/load the model and data
# MODEL_NAME="OpenVLThinker-7B-v1.2"
MODEL_NAME="Qwen2.5-7B-15-Medium"

# Set the image resolution range
MIN_PIXELS = 262144
MAX_PIXELS = 262144

DATASET="openvlthinker_hard_boxed"
OUTPUT_DIR = f'./{DATASET}/{MODEL_NAME}'

prompt_template = "{question}\nPlease reason step by step, and put your final answer within \\boxed{{}}.\n"
# OpenVLThinker-style prompt
# prompt_template = "{question} You FIRST think about the reasoning process as an internal monologue and then provide the final answer."

try:
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"Directory '{OUTPUT_DIR}' created successfully.")
except OSError as e:
    print(f"Error creating directory: {e}")

### Step 1: Preproc the sampling dataset

In [ ]:
# Here we use OpenVLThinker-grpo-hard as an example
import json
import datasets
import random
import base64
from io import BytesIO
from tqdm import tqdm

# Function to directly encode image into base64
def encode_pil_image(pil_img):
    """
    Encode a PIL.Image object to a base64 string with auto-detected format.
    Returns both the base64 string and MIME type (e.g., 'image/png').
    """
    buffered = BytesIO()

    # Auto-detect format from the image or default to PNG
    img_format = pil_img.format or "PNG"
    pil_img.save(buffered, format=img_format)

    img_bytes = buffered.getvalue()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    # Construct MIME type from format (e.g. "image/jpeg")
    mime_type = f"image/{img_format.lower()}"
    return img_b64, mime_type

# data_path = "ydeng9/OpenVLThinker-grpo-hard"
data_path = "ydeng9/OpenVLThinker-grpo-medium"

dataset = datasets.load_dataset(data_path, split="train")

sample_num = 2000

all_prompts = []
for d in tqdm(dataset):
    question = d["problem"].replace("<image>", "").strip()
    images = d["images"]

    assert len(images) == 1, "Error: There are more than 1 images in the images column, which is not expected!"

    img_b64, mime_type = encode_pil_image(images[0])
    
    prompt = prompt_template.format(question=question)
    
    data = {
        "prompt": prompt,
        "img_b64": img_b64,
        "mime_type": mime_type
    }
    
    all_prompts.append(data)

random.shuffle(all_prompts)

with open(SAMPLING_DATASET_PATH, "w") as f:

    json.dump(all_prompts[:sample_num], f, indent=4)

### Step 2: Sample the output from the base model

In [ ]:
# Launch the vllm server to sample the output from the base model
GPUS = "0,1"
cmd = f"./launch_vllm_model_vlm.sh {GPUS} {BASE_MODEL_PATH} {MIN_PIXELS} {MAX_PIXELS}"

print(cmd)

In [ ]:
# After the server is launched, we can use the following code to sample the output from the base model

IS_VLM = True

cmd = f"./sample_data_vllm.sh \
    {BASE_MODEL_PATH} \
    {MODEL_NAME} \
    {SAMPLING_DATASET_PATH} \
    {OUTPUT_DIR} \
    {IS_VLM}"

print(cmd)


In [ ]:
# Get the lvd data
# The vllm serve always use fp16, for lvd, we use fp32 to recover hidden states
GPUS = "4,5,6,7"
LVD_PATH = f"{OUTPUT_DIR}/{MODEL_NAME}.lvd"

cmd = f"./run_get_lvd_embedding_img.sh \
    {LVD_PATH} \
    {BASE_MODEL_PATH} \
    {MIN_PIXELS} \
    {MAX_PIXELS} \
    {GPUS}"
print(cmd)

### Step 3: Initialize hmm with lvd

In [ ]:
import os
from transformers import AutoModelForVision2Seq, AutoTokenizer


# specify the HMM size
HIDDEN_STATES = 4096

# get vocab_size and eos_token_id; might vary for different models #
__tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
__model = AutoModelForVision2Seq.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True)
VOCAB_SIZE = __model.config.vocab_size # will be different to __tokenizer.vocab_size
EOS_TOKEN_ID = __tokenizer.eos_token_id
####################################################################

HMM_MODEL_ID = f'hmm_{MODEL_NAME}_{DATASET}_{HIDDEN_STATES}'
HMM_MODEL_PATH = f'./workspace/models/{HMM_MODEL_ID}'

_ = os.system(f'mkdir -p {HMM_MODEL_PATH}')

In [ ]:
import os

GPUS = '4,5,6,7'
SEQUENCES_FILE = f'{OUTPUT_DIR}/{MODEL_NAME}.lvd'
EMEBEDDINGS_FILE = f'{OUTPUT_DIR}/{MODEL_NAME}.lvd.embeddings.safetensors'

# latent variable distillation
cmd = f'CUDA_VISIBLE_DEVICES={GPUS} python lvd_hmm.py \
    --sequences_file {SEQUENCES_FILE} --embeddings_file {EMEBEDDINGS_FILE} \
    --hidden_states {HIDDEN_STATES} --vocab_size {VOCAB_SIZE} --eos_token_id {EOS_TOKEN_ID} \
    --kmeans_iterations 100 --pseudocount 0.001 \
    --output_file {HMM_MODEL_PATH}/checkpoint-0'
print(cmd)

### Step 4: Train HMM via Expectation Maximization (EM)

In [ ]:
import os

os.system('mkdir -p ./workspace/logs')
LOG_FILE=f'./workspace/logs/{HMM_MODEL_ID}_log.txt'

CUDA_CORES = '4,5,6,7'
TOTAL_CHUNKS = 100
BATCH_SIZE = 512
# BATCH_SIZE = 16
SAVE_PER_STEP = 10
DROPOUT = 0.01

# EM training schedule:
# 1. train for 100 EM steps, each step using 1 chunk of data
# 2. train for 50 EM steps, each step using 2 chunks of data
# 3. train for 40 EM steps, each step using 5 chunks of data
# 4. train for 40 EM steps, each step using 10 chunks of data
# 5. train for 40 EM steps, each step using 20 chunks of data
# 6. train for 10 EM steps, each step using 40 chunks of data
EM_SCHEDULE = "\"100,1;50,2;40,5;40,10;40,20;10,40\""
# EM_SCHEDULE = "\"10,1;5,2;4,5;4,10;4,20;1,40\""

cmd = f'CUDA_VISIBLE_DEVICES={CUDA_CORES} torchrun --standalone --nproc_per_node=gpu train_hmm.py \
    --model_path {HMM_MODEL_PATH} --checkpoint 0 --save_per_step {SAVE_PER_STEP} \
    --data_path {OUTPUT_DIR} --dataset {MODEL_NAME} --total_chunks {TOTAL_CHUNKS} --batch_size {BATCH_SIZE} \
    --em_schedule {EM_SCHEDULE} --dropout {DROPOUT} --log_file {LOG_FILE}'.strip()
print(cmd)